# MAMMAL Model Inference

Run inference with a finetuned MAMMAL model on **any CSV file** containing SMILES strings.

All dataset, dataloader and task logic lives in **`mammal_utils.py`** (same folder).

## Steps
0. Configuration — edit paths here
1. Preview input data
2. Load tokenizer
3. Load model
4. Build DataLoader
5. Run inference
6. Inspect predictions
7. Evaluate (if labels available)
8. Enrichment metrics (Precision@K / EF@K)
9. Top hits

In [ ]:
!pip install biomed-multi-alignment

## 0. Configuration

**Edit the paths below before running.**

In [ ]:
import os
import torch

RUN_ON_WDR91 = False
if RUN_ON_WDR91:
    MODEL_PATH = "michalozeryflato/biomed.omics.bl.sm.ma-ted-458m.wdr91_asms"
    DATA_PATH = "/content/wdr91_test.csv"
    SMILES_COLUMN = "SMILES"
    OUTPUT_PATH = "/content/results/mammal_wdr91_test_predictions.csv"
else:
    MODEL_PATH = "michalozeryflato/biomed.omics.bl.sm.ma-ted-458m.pgk2_del_cdd"
    DATA_PATH = "/content/creative.csv"
    SMILES_COLUMN = "smiles"
    OUTPUT_PATH = "/content/results/mammal_pgk2_del_creative_predictions.csv"

LABEL_COLUMN  = "Label"   

# Base MAMMAL tokenizer / model (HuggingFace hub id or local path)
BASE_MODEL_PATH = "ibm/biomed.omics.bl.sm.ma-ted-458m"

# Inference device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── VALIDATION ─────────────────────────────────────────────────────────────────
# assert os.path.exists(MODEL_PATH), f"Model not found: {MODEL_PATH}"
assert os.path.exists(DATA_PATH),  f"Data not found:  {DATA_PATH}"
os.makedirs(os.path.dirname(OUTPUT_PATH) or ".", exist_ok=True)

print(f"Model : {MODEL_PATH}")
print(f"Data  : {DATA_PATH}")
print(f"Output: {OUTPUT_PATH}")
print(f"Device: {DEVICE}")

## 1. Preview Input Data

In [ ]:
import pandas as pd
input_df = pd.read_csv(DATA_PATH)

print(f"\nLabel distribution:")
print(input_df[LABEL_COLUMN].value_counts().sort_index())

display(input_df)

## 2. Load Tokenizer

In [ ]:
from fuse.data.tokenizers.modular_tokenizer.op import ModularTokenizerOp
tokenizer_op = ModularTokenizerOp.from_pretrained(BASE_MODEL_PATH)
print("Tokenizer loaded.")

## 3. Load Model

In [ ]:
from mammal.model import Mammal
from huggingface_hub import hf_hub_download
from fuse.dl.lightning.pl_module import LightningModuleDefault

base_model = Mammal.from_pretrained(BASE_MODEL_PATH)

checkpoint_path = hf_hub_download(
            repo_id=MODEL_PATH,
            filename="last.ckpt",
            revision="main",
        )
pl_module = LightningModuleDefault.load_from_checkpoint(
        checkpoint_path=checkpoint_path,
        model=base_model,
        model_dir=None,
        map_location="cpu",
    )
model = pl_module._model  # extract the inner Mammal
model.eval().to(DEVICE)

## 4. Build DataLoader

In [ ]:
from torch.utils.data import DataLoader, Dataset
from fuse.data.utils.collates import CollateDefault
from mammal.keys import *
from functools import partial

In [ ]:
class InferenceDataset(Dataset): 
    def __init__(
        self,
        df: pd.DataFrame,
        smiles_column: str,
        tokenizer_op: ModularTokenizerOp,
        drug_max_seq_length: int = 300,
        encoder_input_max_seq_len: int = 320,

    ) -> None:
        self.smiles = df[smiles_column].tolist()
        self.indices = df.index.tolist()
        self.tokenizer_op = tokenizer_op
        self.drug_max_seq_length = drug_max_seq_length
        self.encoder_input_max_seq_len=encoder_input_max_seq_len
        print(f"InferenceDataset: {len(self.smiles)} samples")

    def __len__(self) -> int:
        return len(self.smiles)

    def __getitem__(self, idx: int) -> dict:
        drug_sequence = self.smiles[idx]

        sample: dict = {
            "index": self.indices[idx],
            "data.smiles": drug_sequence,
        }

        # ── Encoder input ──────────────────────────────────────────────────────
        sample[ENCODER_INPUTS_STR] = (
            f"<@TOKENIZER-TYPE=SMILES><SENTINEL_ID_0>"
            f"<MOLECULAR_ENTITY><MOLECULAR_ENTITY_SMALL_MOLECULE>"
            f"<@TOKENIZER-TYPE=SMILES@MAX-LEN={self.drug_max_seq_length}>"
            f"<SEQUENCE_NATURAL_START>{drug_sequence}<SEQUENCE_NATURAL_END><EOS>"
        )
        self.tokenizer_op(
            sample_dict=sample,
            key_in=ENCODER_INPUTS_STR,
            key_out_tokens_ids=ENCODER_INPUTS_TOKENS,
            key_out_attention_mask=ENCODER_INPUTS_ATTENTION_MASK,
            max_seq_len=self.encoder_input_max_seq_len,
        )
        sample[ENCODER_INPUTS_TOKENS] = torch.tensor(sample[ENCODER_INPUTS_TOKENS])
        sample[ENCODER_INPUTS_ATTENTION_MASK] = torch.tensor(
            sample[ENCODER_INPUTS_ATTENTION_MASK]
        )
        return sample

In [ ]:
def build_dataloader(
    df: pd.DataFrame,
    tokenizer_op: ModularTokenizerOp,
    smiles_column: str = "smiles",
    batch_size: int = 128,
    drug_max_seq_length: int = 300,
    encoder_input_max_seq_len:int=320,
    num_workers: int = 0,
) -> DataLoader:
    
    dataset = InferenceDataset(
        df=df,
        smiles_column=smiles_column,
        tokenizer_op=tokenizer_op,
        drug_max_seq_length=drug_max_seq_length,
        encoder_input_max_seq_len=encoder_input_max_seq_len,
    )

    pad_token_id = tokenizer_op.get_token_id("<PAD>")
    special_handlers = {
        ENCODER_INPUTS_TOKENS: partial(
            CollateDefault.crop_padding, pad_token_id=pad_token_id
        ),
        ENCODER_INPUTS_ATTENTION_MASK: partial(
            CollateDefault.crop_padding, pad_token_id=False
        ),
    }

    loader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=CollateDefault(special_handlers_keys=special_handlers),
        shuffle=False,
        num_workers=num_workers,
    )
    return loader

In [ ]:
dataloader = build_dataloader(
    df=input_df,
    tokenizer_op=tokenizer_op,
    smiles_column=SMILES_COLUMN,
)

In [ ]:
first_batch = next(iter(dataloader))

## 5. Run Inference

In [ ]:
from tqdm import tqdm
def run_inference(
    model: Mammal,
    dataloader: DataLoader,
    tokenizer_op: ModularTokenizerOp,
    device: str = "cpu",
    classification_position: int = 1,
) -> pd.DataFrame:

    negative_token_id = tokenizer_op.get_token_id("<0>")
    positive_token_id = tokenizer_op.get_token_id("<1>")
    label_map: dict[Any, int] = {negative_token_id: 0, positive_token_id: 1}

    results: dict[str, list] = {
        "sample_id": [],
        "predicted_label": [],
        "prediction_score": [],
    }

    model.eval()
    model = model.to(device)

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Inference"):
            batch_size = batch[ENCODER_INPUTS_TOKENS].shape[0]

            # Build per-sample dicts for model.generate()
            sample_dicts = [
                {
                    ENCODER_INPUTS_TOKENS: batch[ENCODER_INPUTS_TOKENS][i].to(device),
                    ENCODER_INPUTS_ATTENTION_MASK: batch[ENCODER_INPUTS_ATTENTION_MASK][i].to(device),
                }
                for i in range(batch_size)
            ]

            batch_out = model.generate(
                sample_dicts,
                output_scores=True,
                return_dict_in_generate=True,
                max_new_tokens=5,
            )

            decoder_output = batch_out.get(CLS_PRED, None)  # (B, seq_len)
            decoder_scores = batch_out.get(SCORES, None)  # (B, seq_len, vocab)

            for i in range(batch_size):
                sample_id = batch["index"][i]

                if isinstance(sample_id, torch.Tensor):
                    sample_id = sample_id.item()

                predicted_label: int | None = None
                prediction_score: float | None = None

                if decoder_output is not None and decoder_scores is not None:
                    out_tokens = decoder_output[i].cpu().numpy()
                    out_scores = decoder_scores[i].cpu().numpy()  # (seq_len, vocab)

                    predicted_token = int(out_tokens[classification_position])
                    predicted_label = label_map.get(predicted_token, None)

                    prediction_score = float(
                        out_scores[classification_position, positive_token_id]
                    )

                results["sample_id"].append(sample_id)
                results["predicted_label"].append(predicted_label)
                results["prediction_score"].append(prediction_score)

    return pd.DataFrame(results)


In [ ]:
predictions_df = run_inference(
    model=model,
    dataloader=dataloader,
    tokenizer_op=tokenizer_op,
    device=DEVICE,
)

In [ ]:
predictions_df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved {len(predictions_df)} predictions to: {OUTPUT_PATH}")

In [ ]:
predictions_df = input_df.merge(predictions_df, on="sample_id")
predictions_df = predictions_df.rename(columns={LABEL_COLUMN:"true_label"})

## 6. Inspect Predictions

In [ ]:
print(f"Shape: {predictions_df.shape}")
print(f"\nPredicted label distribution:")
print(predictions_df["predicted_label"].value_counts().sort_index())
print(f"\nScore statistics:")
print(predictions_df["prediction_score"].describe())
display(predictions_df)

## 7. Evaluate (if labels available)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
)

valid   = predictions_df[predictions_df["true_label"].notna()].copy()
y_true  = valid["true_label"].astype(int)
y_pred  = valid["predicted_label"].astype(int)
y_score = valid["prediction_score"].astype(float)

print("=" * 50)
print("CLASSIFICATION METRICS")
print("=" * 50)
print(f"Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision : {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_true, y_pred, zero_division=0):.4f}")
print(f"F1 Score  : {f1_score(y_true, y_pred, zero_division=0):.4f}")
print(f"ROC AUC   : {roc_auc_score(y_true, y_score):.4f}")
print(f"PR AUC    : {average_precision_score(y_true, y_score):.4f}")

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay(cm, display_labels=["Negative", "Positive"]).plot(ax=ax, colorbar=False)
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()


## 8. Enrichment Metrics

Precision@K and Enrichment Factor (EF@K).

**EF@K** = Precision@K / hit_prevalence  
where hit_prevalence = fraction of positives in the full dataset.

In [ ]:
import numpy as np
from mammal_utils import calculate_enrichment_metrics

n_total = len(valid)
top_k_values = sorted({k for k in [10, 50, 100, 500, 1000, 2000] if k <= n_total})
if n_total not in top_k_values:
    top_k_values.append(n_total)

enrichment = calculate_enrichment_metrics(
    y_true.values, y_score.values, top_k_values=top_k_values
)

n_positives = int(y_true.sum())
hit_prev    = n_positives / n_total

print("=" * 60)
print("ENRICHMENT METRICS")
print("=" * 60)
print(f"Dataset : {n_total} compounds  |  Positives: {n_positives}  |  Hit rate: {hit_prev:.2%}")
print()
print(f"{'K':>8}  {'Precision@K':>12}  {'EF@K':>8}  {'Hits in top-K':>14}")
print("-" * 50)
for k in top_k_values:
    prec = enrichment.get(f"Precision@{k}", np.nan)
    ef   = enrichment.get(f"EF@{k}",        np.nan)
    hits = int(round(prec * k)) if not np.isnan(prec) else "N/A"
    print(f"{k:>8}  {prec:>12.4f}  {ef:>8.2f}  {hits:>14}")
print("=" * 60)

ef_rows = [
    {
        "K":             k,
        "Precision@K":   enrichment.get(f"Precision@{k}", np.nan),
        "EF@K":          enrichment.get(f"EF@{k}",        np.nan),
        "Hits_in_top_K": int(round(enrichment.get(f"Precision@{k}", 0) * k)),
    }
    for k in top_k_values
]
ef_df = pd.DataFrame(ef_rows)
ef_df


## 9. Top Hits

In [ ]:
def calculate_enrichment_metrics(
    y_true,
    y_scores,
    top_k_values: list[int] | None = None,
) -> dict:
    if top_k_values is None:
        top_k_values = [10, 50, 100, 500, 1000, 2000]

    y_true = np.asarray(y_true)
    y_scores = np.asarray(y_scores)

    sorted_indices = np.argsort(y_scores)[::-1]
    sorted_labels = y_true[sorted_indices]

    hit_prevalence = np.sum(y_true) / len(y_true)

    enrichment: dict = {}
    for k in top_k_values:
        if k <= len(sorted_labels):
            top_k_labels = sorted_labels[:k]
            precision_at_k = np.sum(top_k_labels) / k
            enrichment[f"Precision@{k}"] = precision_at_k
            enrichment[f"EF@{k}"] = (
                precision_at_k / hit_prevalence if hit_prevalence > 0 else np.nan
            )
        else:
            enrichment[f"Precision@{k}"] = np.nan
            enrichment[f"EF@{k}"] = np.nan

    return enrichment

In [ ]:
top_hits = (
    predictions_df[predictions_df["predicted_label"] == 1]
    .sort_values("prediction_score", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

n_pos = (predictions_df["predicted_label"] == 1).sum()
print(f"Top 20 predicted positives (out of {n_pos} total predicted positives):")
top_hits[["sample_id", "smiles", "prediction_score", "true_label"]]